# RadarMD — full training on Kaggle (free GPU)

Trains on the **complete NIH ChestX-ray14** dataset (112k images) using Kaggle's
free P100 / T4×2 GPU. The full dataset is mounted read-only at
`/kaggle/input/data`, so there is **nothing to download** — no Kaggle token, no
Drive, no sharding. All real logic lives in `src/radarmd`; this notebook only
clones, installs, and launches `scripts/train.py`.

## One-time setup before Run All
1. **Add the data**: right panel → *Add Input* → search **"NIH Chest X-rays"**
   (`nih-chest-xrays/data`) → Add. It mounts at `/kaggle/input/data`.
2. **GPU**: *Settings → Accelerator → GPU T4 x2* (or P100).
3. **Internet**: *Settings → Internet → On* (needed for pip + git + W&B).
4. **W&B key (optional but recommended)**: *Add-ons → Secrets* → add
   `WANDB_API_KEY`. Without it, set `WANDB_MODE=disabled` in the training cells.

## 1. Clone + install

In [ ]:
!git clone https://github.com/aarushnarang02/radarmd.git
%cd radarmd
# Kaggle ships a CUDA torch; install the rest of the project.
!pip install -q -e '.[dev,data,track,interpret]'

## 2. W&B login (skip if you didn't add the secret)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    WANDB = 'online'
    print('W&B key loaded.')
except Exception as e:
    WANDB = 'disabled'
    print(f'No W&B secret ({e}); logging to CSV instead.')

## 3. Point at the mounted dataset

The full dataset mounts at `/kaggle/input/data` with `Data_Entry_2017.csv` at the
root and images under `images_001/images/ … images_012/images/`. The DataModule's
recursive image index handles that layout as-is.

In [ ]:
import os, glob
# Kaggle's mount path for the NIH dataset varies (sometimes /kaggle/input/data,
# sometimes a deeper /kaggle/input/datasets/organizations/...). Locate the
# dataset root by finding Data_Entry_2017.csv wherever it landed.
hits = glob.glob('/kaggle/input/**/Data_Entry_2017.csv', recursive=True)
assert hits, 'Add the official nih-chest-xrays/data dataset first (Add Input).'
META = hits[0]
DATA = os.path.dirname(META)
print('dataset root:', DATA)
print('images found:', len(glob.glob(f'{DATA}/**/*.png', recursive=True)))
OUT = '/kaggle/working/outputs'

## 4a. Baseline for the '6-point gain' (frozen-ish, low-res ResNet-50)

Short, low-resolution run whose mean AUROC is the number the tuned DenseNet is
measured against.

In [ ]:
!python scripts/train.py --config configs/base.yaml --out $OUT/baseline \
    model.backbone=resnet50 data.image_size=224 optim.max_epochs=8 \
    data.data_dir=$DATA data.metadata_csv=$META data.num_workers=4 \
    trainer.precision=16-mixed \
    wandb.mode=$WANDB wandb.project=radarmd

## 4b. Headline run: tuned DenseNet-121 (CheXNet-style)

In [ ]:
!python scripts/train.py --config configs/densenet121.yaml --out $OUT/densenet121 \
    data.data_dir=$DATA data.metadata_csv=$META data.num_workers=4 \
    wandb.mode=$WANDB wandb.project=radarmd

## 4c. ConvNeXt-Tiny (second backbone)

In [ ]:
!python scripts/train.py --config configs/convnext.yaml --out $OUT/convnext \
    data.data_dir=$DATA data.metadata_csv=$META data.num_workers=4 \
    wandb.mode=$WANDB wandb.project=radarmd

## 5. Evaluate the best checkpoint (thresholds + sensitivity floor)

Tunes per-class operating points on val and reports test AUROC/AP/sensitivity/
specificity/F1, checking the 'missed diagnoses < 8%' target on critical findings.

In [ ]:
import glob
ckpts = sorted(glob.glob(f'{OUT}/densenet121/checkpoints/*.ckpt'))
print('checkpoints:', ckpts)
BEST = ckpts[-1]
!python scripts/evaluate.py --checkpoint "$BEST" \
    --data-dir $DATA --metadata $META --out $OUT/evaluation

## 6. Grad-CAM localization vs the 880 ground-truth boxes

In [ ]:
!python scripts/evaluate_localization.py --checkpoint "$BEST" \
    --data-dir $DATA --bbox-csv $DATA/BBox_List_2017.csv \
    --out $OUT/localization/localization.csv

## 7. Export ONNX + save artifacts

Everything under `/kaggle/working` can be saved as a **Kaggle dataset** (or
downloaded) so we can pull `radarmd.onnx` + `thresholds.json` locally for the
FastAPI service and Cloud Run deploy.

In [ ]:
!python scripts/export_onnx.py --checkpoint "$BEST" \
    --out $OUT/radarmd.onnx --image-size 320
!ls -lh $OUT $OUT/evaluation